# KV quantization gate: does routing survive low-bit keys and values

Question: does the `read` relevance signal (Project C) survive when the KV cache is stored at
8, 6, 4, 3 or 2 bits? If routing holds at every bit width there is nothing for a salience signal
to guide. If it collapses at the highest width tested there is no routing left to preserve. A
cliff in between is the regime where salience-guided bit assignment could matter.
See [kvquant_gate.md](../docs/kvquant_gate.md).

## What runs

`declare/run.py --modes read --kv-bits B --kv-scheme S`. Keys and values are fake-quantized
inside attention (simulated, not packed), so this measures accuracy loss only, not memory.

- `token`: keys and values per token, groups of 32 along the head dimension.
- `channel_keys`: keys per channel over groups of 32 tokens, values per token. This is the
  KIVI-style layout, with the trailing partial group left at full precision and padded
  positions excluded from each group's range.

Conditions per model and seed: one full-precision baseline plus 2 schemes x 5 bit widths = 11
runs. Models: Qwen2.5-0.5B and 1.5B. Probes: the same 384 as the routing table (96 trajectories
x 4 probes), with layouts pinned to `results/declare/gold/gold384_s<seed>.json`.

## Guards

1. Layouts must match the committed gold distribution for the seed or the run refuses to start.
2. The full-precision baseline must reproduce the committed `results/declare` hit rate for that
   model and seed to within 0.01. If it does not, the quantized numbers are not comparable and
   the notebook stops.
3. The report marks a baseline invalid if it is below 2 sigma over the best constant control.

## Budget

Unmeasured. `read` is one batched forward per probe and shuffle, so each run should take a few
minutes on a T4. Run one seed per session and keep `runs_kvquant.zip` between sessions.
Measure the first run's wall time before queueing more.


In [ ]:
import glob, os, subprocess, sys, zipfile

GIT_URL = 'https://github.com/rajul-kk/context-to-weights.git'
REPO = '/kaggle/working/myrios'
MARKER = 'baselines/cascading.py'
FRESH = True
REQUIRE = [('declare/run.py', 'kv-scheme'), ('eval/kvquant_report.py', 'paired_t')]

def looks_like_source(d):
    return os.path.exists(os.path.join(d, MARKER))

os.makedirs('/kaggle/working', exist_ok=True)
os.chdir('/kaggle/working')

if FRESH and GIT_URL and os.path.exists(REPO):
    subprocess.run(['rm', '-rf', REPO], check=True)
    print(f'removed {REPO} so the clone is current')

if not looks_like_source(REPO) and GIT_URL:
    r = subprocess.run(['git', 'clone', '--depth', '1', GIT_URL, REPO],
                       capture_output=True, text=True)
    print(r.stdout, r.stderr)

if not looks_like_source(REPO):
    print('no clone; falling back to attached inputs')
    src = None
    for d in sorted(glob.glob('/kaggle/input/*')):
        if looks_like_source(d):
            src = d
            break
        for z in sorted(glob.glob(os.path.join(d, '*.zip'))):
            os.makedirs(REPO, exist_ok=True)
            zipfile.ZipFile(z).extractall(REPO)
            if looks_like_source(REPO):
                src = REPO
                break
        if src:
            break
    if src and src != REPO:
        subprocess.run(['cp', '-r', src, REPO], check=True)

assert looks_like_source(REPO), (
    'No source found. Either (a) set GIT_URL above, or (b) run scripts/package_source.py '
    'locally, upload the zip as a Kaggle Dataset, and attach it via Add Input. '
    f'Searched /kaggle/input/*, saw: {sorted(glob.glob("/kaggle/input/*"))}')

os.chdir(REPO)
sys.path.insert(0, REPO)

_h = subprocess.run(['git', 'log', '--oneline', '-1'], capture_output=True, text=True, cwd=REPO)
print('source commit:', _h.stdout.strip() or 'unknown (not a git checkout)')

_stale = [f for f, token in REQUIRE
          if not os.path.exists(f) or token not in open(f, encoding='utf-8').read()]
assert not _stale, (
    f'This checkout is older than the seed support these cells need: {_stale}. '
    'It came from an attached dataset or a leftover /kaggle/working/myrios rather than a '
    'fresh clone. Detach any source dataset under Add Input, keep only the runs_declare.zip '
    'dataset, and re-run this cell with FRESH = True.')
print('source is current for the kv quantization grid')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'peft', 'accelerate', 'datasets'])

try:
    import importlib.metadata as _md
    _v = _md.version('torchao')
    if tuple(int(x) for x in _v.split('.')[:2]) < (0, 16):
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'uninstall', '-y', 'torchao'])
        print(f'removed incompatible torchao {_v}')
except Exception:
    pass

exec(open('notebooks/_runner.py').read())
print('cwd', os.getcwd())
print('files', sorted(os.listdir('.'))[:10])
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())
if not torch.cuda.is_available():
    print()
    print('=' * 68)
    print('NO GPU. Kaggle installed the CPU build of torch, so this session')
    print('has no accelerator attached. Everything below will be far too slow.')
    print()
    print('Fix: right panel -> Session options -> Accelerator -> GPU T4 x2,')
    print('then Run All again. The image swaps to a CUDA torch build on restart.')
    print('=' * 68)
else:
    print('gpu', torch.cuda.get_device_name(0),
          f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


Re-run only the next cell if the kernel restarts; it re-derives everything.

- `SEEDS`: one seed per session
- `MODELS`, `BITS`, `SCHEMES`: the grid
- `TOLERANCE`: allowed gap between the baseline and the committed result


In [ ]:
import os, sys
REPO = '/kaggle/working/myrios'
os.chdir(REPO); sys.path.insert(0, REPO)
if 'run' not in dir():
    exec(open('notebooks/_runner.py').read())

CFG = 'configs/kaggle_kvquant.yaml'
RUNS = '/kaggle/working/artifacts/runs_kvquant'
DATA = 'artifacts/data/hotpotqa_spread'
N_EVAL = 96
PER_TRAJECTORY = 4
SEEDS = [0]
MODELS = ['Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct']
BITS = [8, 6, 4, 3, 2]
SCHEMES = ['token', 'channel_keys']
TOLERANCE = 0.01

from common.io import load_config
_c = load_config(CFG)
assert _c['run_root'] == RUNS, f"run_root mismatch: {_c['run_root']} != {RUNS}"
assert _c['data']['dir'] == DATA, f"data.dir mismatch: {_c['data']['dir']} != {DATA}"
os.makedirs(RUNS, exist_ok=True)

conditions = [(0, 'none')] + [(b, s) for s in SCHEMES for b in BITS]
print(f'seeds {SEEDS}, models {len(MODELS)}, conditions per run {len(conditions)}')
print(f'total runs this session: {len(SEEDS) * len(MODELS) * len(conditions)}')


Restore `runs_kvquant.zip` from the previous session if one exists, so finished runs are
skipped and the report merges them. Attach it as an input dataset and set `ARCHIVE`.


In [ ]:
ARCHIVE = '/kaggle/input/CHANGE-ME/runs_kvquant.zip'
import os
if os.path.exists(ARCHIVE):
    run(f"python scripts/kaggle_sync.py restore --archive {ARCHIVE} --run-root {RUNS}")
else:
    print(f'no archive at {ARCHIVE}; starting from scratch')


In [ ]:
run(f"python data/load_hotpotqa.py --n-train 8 --n-eval {N_EVAL} --per-trajectory {PER_TRAJECTORY} --early-frac 1.0 --out {DATA}")


## The grid

Each run writes `read.summary.json` with `kv_bits` and `kv_scheme`. The baseline for a model
and seed runs first and is checked against the committed result before any quantized run.


In [ ]:
import json
from pathlib import Path

def slug_of(m):
    return m.split('/')[-1]

def tag(slug, seed, bits, scheme):
    return f'{slug}_{scheme}{bits}_s{seed}'

def committed_hit(slug, seed):
    name = slug if seed == 0 else f'{slug}_s{seed}'
    path = Path(REPO) / 'results' / 'declare' / name / 'read.summary.json'
    return json.load(open(path))['hit_rate']

total = len(SEEDS) * len(MODELS) * len(conditions)
i = 0
for seed in SEEDS:
    gold = f'{REPO}/results/declare/gold/gold384_s{seed}.json'
    for m in MODELS:
        for bits, scheme in conditions:
            i += 1
            out = f'{RUNS}/{tag(slug_of(m), seed, bits, scheme)}'
            if os.path.exists(f'{out}/read.summary.json'):
                print(f'[{i}/{total}] skip {out}')
                continue
            print('=' * 70)
            print(f'[{i}/{total}] {slug_of(m)} seed {seed} bits {bits} {scheme}', flush=True)
            flags = f'--kv-bits {bits} --kv-scheme {scheme}' if bits else ''
            run(f"python declare/run.py --config {CFG} --modes read {flags} --out {out} "
                f"--gold-ref {gold} --set model.base={m} seed={seed}")
            if bits == 0:
                got = json.load(open(f'{out}/read.summary.json'))['hit_rate']
                want = committed_hit(slug_of(m), seed)
                print(f'baseline hit {got:.4f} vs committed {want:.4f}')
                if abs(got - want) > TOLERANCE:
                    raise RuntimeError(
                        f'baseline {got:.4f} does not reproduce the committed {want:.4f} for '
                        f'{slug_of(m)} seed {seed}; the quantized runs would not be comparable')
            run(f"python scripts/kaggle_sync.py save --run-root {RUNS} "
                f"--archive /kaggle/working/runs_kvquant.zip")


## Report


In [ ]:
run(f"python eval/kvquant_report.py --run-root {RUNS} --out docs/results_kvquant.md")
from IPython.display import Markdown, display
display(Markdown(open('docs/results_kvquant.md').read()))
run(f"python scripts/kaggle_sync.py save --run-root {RUNS} --archive /kaggle/working/runs_kvquant.zip")
print()
print('=' * 64)
print('DOWNLOAD /kaggle/working/runs_kvquant.zip (right panel -> Output)')
print('=' * 64)
